In [15]:
from pydantic import BaseModel
import uuid

class Task(BaseModel):
    """Task class to hold task information."""
    task_id: str | None = None
    task_status: str = "initiated"
    task_description: str | None = None
    task_name: str
    task_parameters: Dict[str, Any] | None = None
    task_state: Dict[str, Any] | None = None

    def __init__(
            self, 
            task_name: str, 
            task_description = None, 
            task_id: str = None, 
            task_parameters: Dict[str, Any] = None,
            **kwargs
        ) -> None:
        if not task_id:
            task_id = str(uuid.uuid4())
        task_parameters["ext_args"]=kwargs
        task_state=task_parameters["ext_args"].get("task_state",None)
        super().__init__(
            task_id=task_id,
            task_status="pending" if task_id else "initiated",
            task_name=task_name,
            task_description=task_description,
            task_parameters=task_parameters,
        )
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            "task_id": self.task_id,
            "task_status": self.task_status,
            "task_name": self.task_name,
            "task_description": self.task_description,
            "task_parameters": self.task_parameters,
            "task_state": self.task_state
        }
    
    def __str__(self) -> str:
        return f"Task: {self.task_name} ({self.task_id}) - {self.task_status}"


In [18]:
task_dict = {"task_name":"extract_data","task_parameters":{"source_content_id":"fc_12345"}}

In [19]:
task = Task(**task_dict)

In [20]:
print(task.task_parameters)

{'source_content_id': 'fc_12345', 'ext_args': {}}


In [1]:
from mcp.server.fastmcp import FastMCP
from typing import List
from autogen_ext.agents.probill import ProbillPlanAgent
from autogen_ext.models.ollama import OllamaChatCompletionClient
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelFamily
from autogen_agentchat.base import TaskResult
from pydantic import BaseModel
from autogen_agentchat.ui import Console
model_info = {
    "vision": False,
    "function_calling": True,
    "json_output": True,
    "family": ModelFamily.UNKNOWN
}

class Answer(BaseModel):
    thought: str
    topic: str
    answer: str

class Answer_Q(BaseModel):
    """
    Answer_Q represents a detailed answer response structure.
    
    It includes:
        - thought: your reasoning process,
        - topic: the subject matter being addressed,
        - final_answer: the final main response content, use MarkDown to answer
        - suggestion_topic_for_user: a subsequent topic/question for further clarification.
    """
    thought: str
    topic: str
    final_answer: str
    suggestion_topics_for_user: List[str]

model_qwen = "qwen2.5-coder:32b-instruct-q5_0"
model_qwq ="qwq:32b-fp16"
model_client_1=OpenAIChatCompletionClient(
    model=model_qwq, 
    api_key="sk-xxx",
    base_url="http://10.0.40.100:11434/v1",
    model_info=model_info,
    # response_format=Answer
)

model_client_2=OpenAIChatCompletionClient(
    model=model_qwq, 
    api_key="sk-xxx",
    base_url="http://10.0.40.100:11434/v1",
    model_info=model_info,
    response_format=Answer_Q
)

plan_agent = ProbillPlanAgent("proboll_planner", model_client=model_client_1, model_client_stream=True)
response = await Console(plan_agent.run_stream(task="My name is Phil, How are you? Don't over thinking."))
print("\n\n------------\n")
print("The complete response:", flush=True)
for msg in response.messages:
    print(msg.source, flush=True)    
    print(msg.content, flush=True)
    print("Usage: ",msg.models_usage, flush=True)
print("\n\n------------\n")


---------- user ----------
My name is Phil, How are you? Don't over thinking.
My name is Phil, How are you? Don't over thinking.
---------- proboll_planner ----------
<think>

Okay, the user is Phil and he's asking how I am. He mentioned not to overthink it. Let me keep it simple.

I should respond politely, state that I'm doing well, and maybe ask him back how he is. Keep it friendly and straightforward. No need for any extra details since he said not to overcomplicate things. Just a basic greeting response.
</think>

Hello Phil! I'm just a computer program, but I'm here to help you however I can. How can I assist you today?


------------

The complete response:
user
My name is Phil, How are you? Don't over thinking.
Usage:  None
proboll_planner
<think>

Okay, the user is Phil and he's asking how I am. He mentioned not to overthink it. Let me keep it simple.

I should respond politely, state that I'm doing well, and maybe ask him back how he is. Keep it friendly and straightforward. 

In [2]:
from autogen_core.models import UserMessage
from autogen_core import CancellationToken
from typing import Callable
import asyncio

messages = [
    UserMessage(content="Write a very short story about a dragon in Chinese.", source="user"),
]

messages = [
    UserMessage(content="Do you know my name?.", source="user"),
]


def sample_function(**kwargs) -> str:
    print("Cancelled by user")
    print(kwargs)
    return f"Sample Callable received: {kwargs}"

# Assign the function to a variable of type Callable.
sample_callable: Callable[[str], str] = sample_function

# Demonstrate usage of the callable.
result = sample_callable(msg="Hello, World!")
print(result)

def _cancelled(**kwargs)->Callable:
    
    print(**kwargs)

cancellation_token = CancellationToken()
cancellation_token.add_callback(sample_callable)
# Create a stream.
stream = model_client_2.create_stream(messages=messages, json_output=True, cancellation_token=cancellation_token)
# Iterate over the stream and print the responses.
print("Streamed responses:")
words = 0
try:
    async for response in stream:  # type: ignore
        if isinstance(response, str):
            # A partial response is a string.
            print(response, flush=True, end="")
        else:
            # The last response is a CreateResult object with the complete message.
            print("\n\n------------\n")
            print("The complete response:", flush=True)
            print(response.content, flush=True)
            print("\n\n------------\n")
            print("The token usage was:", flush=True)
            print(response.usage, flush=True)
except asyncio.CancelledError as error:
    print(f"\nStream cancelled due to cancellation: {error}", flush=True)
except Exception as error:
    print(f"\nStream cancelled due to error: {error}", flush=True)



Cancelled by user
{'msg': 'Hello, World!'}
Sample Callable received: {'msg': 'Hello, World!'}
Streamed responses:
{
"thought": "Okay, the user is asking if I know their name. Let me check how they addressed me. The conversation history shows they mentioned 'my name' but didn't provide it explicitly. Since they haven't told me their name yet, I should respond that I don't know it and offer to help once they share it.",
"topic": "The user’s name",
"final_answer": "Hello! I'm Qwen. I don't currently know your name since you haven't shared it with me yet. Feel free to tell me what your name is, and I can greet you properly!"
 		                 ,
 "suggestion_topics_for_user":[
        "Sharing personal information",
        "Introduction"
    ]
}

------------

The complete response:
{
"thought": "Okay, the user is asking if I know their name. Let me check how they addressed me. The conversation history shows they mentioned 'my name' but didn't provide it explicitly. Since they haven't to

In [3]:
response = await Console(plan_agent.run_stream(task="Do you know my name?"))
print("\n\n------------\n")
print("The complete response:", flush=True)
for msg in response.messages:
    print(msg.source, flush=True)    
    print(msg.content, flush=True)
    print("Usage: ",msg.models_usage, flush=True)
print("\n\n------------\n")

---------- user ----------
Do you know my name?
Do you know my name?
---------- proboll_planner ----------
<think>

Alright, the user asked if I know his name. Let me check the previous messages.

Looking back at the history, the first message was "My name is Phil, How are you? Don't over thinking." So he explicitly stated his name there. 

I should confirm that I remember his name. The response needs to be straightforward and personal. Maybe say something like yes, I remember he mentioned it earlier, then mention him by name to make it personal. Keep the tone friendly and helpful as before.
</think>

Yes, Phil! You told me your name in our first message. Nice to chat with you again! How can I assist you today?


------------

The complete response:
user
Do you know my name?
Usage:  None
proboll_planner
<think>

Alright, the user asked if I know his name. Let me check the previous messages.

Looking back at the history, the first message was "My name is Phil, How are you? Don't over th

In [6]:
plan_agent._model_client.dump_component()

ComponentModel(provider='autogen_ext.models.openai.OpenAIChatCompletionClient', component_type='model', version=1, component_version=1, description='Chat completion client for OpenAI hosted models.', label='OpenAIChatCompletionClient', config={'model': 'qwq', 'api_key': 'sk-xxx', 'model_info': {'vision': False, 'function_calling': True, 'json_output': True, 'family': 'unknown'}, 'base_url': 'http://10.0.40.49:11434/v1'})

In [7]:
_component_plan_agent = plan_agent.dump_component()
component_json = _component_plan_agent.model_dump_json(indent=2)
print(component_json)

{
  "provider": "autogen_ext.agents.probill.ProbillPlanAgent",
  "component_type": "agent",
  "version": 1,
  "component_version": 1,
  "description": "An agent, used by Probill that provides plaaning service.",
  "label": "ProbillPlanAgent",
  "config": {
    "name": "proboll_planner",
    "model_client": {
      "provider": "autogen_ext.models.openai.OpenAIChatCompletionClient",
      "component_type": "model",
      "version": 1,
      "component_version": 1,
      "description": "Chat completion client for OpenAI hosted models.",
      "label": "OpenAIChatCompletionClient",
      "config": {
        "model": "qwq",
        "api_key": "sk-xxx",
        "model_info": {
          "vision": false,
          "function_calling": true,
          "json_output": true,
          "family": "unknown"
        },
        "base_url": "http://10.0.40.49:11434/v1"
      }
    },
    "tools": [],
    "model_context": {
      "provider": "autogen_core.model_context.UnboundedChatCompletionContext",
  

In [9]:
print(component_json)
print(type(component_json))

{'provider': 'autogen_ext.agents.probill.ProbillPlanAgent', 'component_type': 'agent', 'version': 1, 'component_version': 1, 'description': 'An agent, used by Probill that provides plaaning service.', 'label': 'ProbillPlanAgent', 'config': {'name': 'proboll_planner', 'model_client': {'provider': 'autogen_ext.models.ollama.OllamaChatCompletionClient', 'component_type': 'model', 'version': 1, 'component_version': 1, 'description': 'Chat completion client for Ollama hosted models.', 'label': 'OllamaChatCompletionClient', 'config': {'model': 'qwq', 'host': 'http://10.0.40.49:11434', 'response_format': <class '__main__.Answer'>, 'follow_redirects': True}}, 'tools': [], 'model_context': {'provider': 'autogen_core.model_context.UnboundedChatCompletionContext', 'component_type': 'chat_completion_context', 'version': 1, 'component_version': 1, 'description': 'An unbounded chat completion context that keeps a view of the all the messages.', 'label': 'UnboundedChatCompletionContext', 'config': {}

In [4]:
component_json.get("config", {}).get("model_client",{}).get("config")["model_info"]=model_client.model_info

In [10]:
import json
print(json.dumps(component_json, indent=2))

TypeError: Object of type ModelMetaclass is not JSON serializable

In [ ]:
model_cpn = model_client.dump_component()

print(model_cpn.model_dump_json(indent=2))

{
  "provider": "autogen_ext.models.ollama.OllamaChatCompletionClient",
  "component_type": "model",
  "version": 1,
  "component_version": 1,
  "description": "Chat completion client for Ollama hosted models.",
  "label": "OllamaChatCompletionClient",
  "config": {
    "model": "qwq",
    "host": "http://10.0.40.49:11434",
    "follow_redirects": true
  }
}


In [5]:
model_client.model_info

{'vision': False,
 'function_calling': True,
 'json_output': True,
 'family': 'r1'}

In [20]:
temp = """{
  "provider": "autogen_ext.models.ollama.OllamaChatCompletionClient",
  "component_type": "model",
  "version": 1,
  "component_version": 1,
  "description": "Chat completion client for Ollama hosted models.",
  "label": "OllamaChatCompletionClient",
  "config": {
    "model": "qwq",
    "model_info": {
        "vision": false,
        "function_calling": true,
        "json_output": true,
        "family": "r1"
    },
    "host": "http://10.0.40.49:11434",
    "follow_redirects": true
  }
}"""

temp_json = json.loads(temp)

In [21]:
print(json.dumps(temp_json, indent=2))

{
  "provider": "autogen_ext.models.ollama.OllamaChatCompletionClient",
  "component_type": "model",
  "version": 1,
  "component_version": 1,
  "description": "Chat completion client for Ollama hosted models.",
  "label": "OllamaChatCompletionClient",
  "config": {
    "model": "qwq",
    "model_info": {
      "vision": false,
      "function_calling": true,
      "json_output": true,
      "family": "r1"
    },
    "host": "http://10.0.40.49:11434",
    "follow_redirects": true
  }
}


In [ ]:
from autogen_agentchat.teams import MagenticOneGroupChat